# Step1 导入相关模块


In [ ]:
# md 文档编辑好后esc - m - shift enter

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM,DataCollatorForSeq2Seq, TrainingArguments,Trainer, AutoModel

In [ ]:
import datasets
datasets.__version__

In [ ]:
import transformers
transformers.__version__


In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Step2 加载数据集

In [ ]:
ds = load_dataset('json', data_dir="../alpaca_data_zh/")
ds = ds['train'] # 加载数据集后放到train数据集中
ds

In [ ]:
# 使用切面查看部分样本数据
ds[:2]

# Step3 数据预处理

In [ ]:
# 分词器,langbot
# AutoTokenizer 读取指定模型中文件
# 需要梯子后才能访问下载huggle face

model_name = 'Langboat/bloom-1b4-zh'

model_path_local = r"D:\models\bloom-1b4-zh"


tokenizer = AutoTokenizer.from_pretrained(model_path_local)
tokenizer

In [ ]:
# 使用分词器读取内容并处理样本
def process_func(example):
    # 单条样本最大长度
    MAX_LENGTH = 256
    # 将文本数据交给模型后转为模型能看得懂的数据，模型关注id;transformers相关的大模型都要计算【注意力机制】，所以有attention_mask；
    input_ids, attention_mask, labels = [], [], []
    # 使用tokenizer做分词, 告诉模型你作为Assistant开始给我回答
    instruction = tokenizer("\n".join(["Human:"+example['instruction'], example['input']]).strip() + "\n\nAssistant:")
    # eos_token为结束符，表示这句话说完了
    response = tokenizer(example['output'] + tokenizer.eos_token)

    input_ids = instruction['input_ids'] + response['input_ids']
    attention_mask = instruction['attention_mask'] + response['attention_mask']
    # 只关心后半段 response['input_ids']
    labels = [-100] * len(instruction['input_ids']) + response['input_ids']

    if len(input_ids)> MAX_LENGTH:
        # 大于长度后，对数据做切片操作
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return{
        'input_ids':input_ids, 'attention_mask':attention_mask, 'labels':labels
    }
    

In [ ]:
# 调用函数处理每条样本数据
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

In [ ]:
# 截取部分数据查看
tokenized_ds[:2]

# 解码
tokenizer.decode(tokenized_ds[:2]['input_ids'])

In [ ]:
# 解码 labels中不等于 -100 的内容
tokenizer.decode(list(filter(lambda x: x!=-100, tokenized_ds[2]['labels'])))

# Step4 模型创建

In [ ]:
# 导入模型
# model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage = True)
# model

# 导入本地模型
model = AutoModelForCausalLM.from_pretrained(model_path_local, low_cpu_mem_usage = True)

In [ ]:
# 当前什么设备
print(model.device)

# 汇总参数量
sum(param.numel() for param in model.parameters())

# 根据参数量计算需要的显存

 ** float32 用4个字节存储 **
model size: 1.36 * 4 ≈ 5.26

 **做全量参数调参 **
SGD gradient ： 1.36 * 4 ≈ 5.26
with Momentum Optimizer : 1.36 * 4 ≈ 5.26
with Adam Optimizer: 还有计算二阶动量 : 1.36 * 4 ≈ 5.26

 ** 总共 **
1.36 * 4 * 4 ≈ 20.8G

# LoRA

训练的内容与原始模型中的权重独立开，为了保证训练参数中内容与原始模型中参数数量尽量靠近，将每一批需要训练的参数拆分为两个较小的矩阵

## LoRA Step1 配置文件

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM
    #target_modules=["query_key_value", "dense_4h_to_h"]
)
print(config)

# LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, 
# peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, 
# peft_version='0.19.1', base_model_name_or_path=None, revision=None, 
# inference_mode=False, r=8, target_modules=None, exclude_modules=None, 
# lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none',
#  use_rslora=False, modules_to_save=None, init_lora_weights=True, 
# layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, 
# megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None,
#  loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None,
#  use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, 
# layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False),
#  lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## LoRA Step2 创建模型

In [ ]:
print(type(model))
print(model is not None)

model = get_peft_model(model, config)
model

In [ ]:
config

In [ ]:
# 查看prefix tuning 模型中配置项,跟随配置中prefix_projection的值变化而改变，False为不使用全链接，True表示使用全连接
model.prompt_encoder

In [ ]:
# 可训练的参数量，参数量与使用是否有全连接层来判断
# 使用全连接时，参数量将急剧增加
model.print_trainable_parameters()

# Step5 配置训练参数

In [ ]:
args = TrainingArguments(
    output_dir='./chatbot', # 输出文件夹，存储模型预测结果和模型文件checkpoints
    per_device_train_batch_size=1, # 训练模型时，每个GPU核/CPU 上面对应的batchSize大小（样本数）
    gradient_accumulation_steps=8, # 执行反向传播/更新参数前，对应梯度计算累计了 n 次
    logging_steps=10, # 每隔 n 次迭代，落地一次日志
    num_train_epochs=1 # 模型学习数据集学习 n 遍
)


# Step6 创建训练器

In [ ]:
# 获取训练器
trainer = Trainer(
    model= model, # 虽然传的对象，但部分参数已经被冻结了
    args= args,
    train_dataset= tokenized_ds,
    data_collator= DataCollatorForSeq2Seq(tokenizer=tokenizer, padding = True), # 构建一批次数据
)

In [ ]:
# 开始训练
trainer.train()

# 加载训练好的PEFT模型

In [ ]:
from peft import PeftModel

# 传入的model内容为原始模型,训练好的向量
peft_model = PeftModel.from_pretrained(model=model, model_id = './checkpoint')

# STEP8 模型推理

In [ ]:
model.device
# 如果使用的cpu进行训练，可指定使用gpu
model = model.cuda()

In [ ]:
ipt = tokenizer("Human:{}\n{}". format("如何提高学习效率？","").strip() + "\n\nAssistant:", 
                return_tensor = 'pt').to(model.device)
# model输出转为文本
tokenizer.decode(model.generate(**ipt, max_length = 256, do_sample=True)[0], skip_special_tokens=True)